# Diversity and Inclusion Dashboard Analysis

## Project Overview
Analyze workforce demographic data across levels, teams, promotions, and attrition to surface diversity and inclusion patterns.

## Learning Objectives
- Calculate representation metrics across organizational levels
- Analyze promotion and attrition rates by demographic groups
- Identify pipeline leakage points where representation drops
- Build a comprehensive D&I analytics framework

## Problem Statement
Leadership wants a data-driven view of workforce diversity: where representation gaps exist, whether advancement is equitable, and whether attrition disproportionately affects certain groups.

## Why This Project Matters
Diverse organizations outperform peers on innovation and financial results. D&I analytics helps identify systemic gaps and track progress toward equitable outcomes.

## Dataset Overview
Synthetic workforce dataset: ~5,000 employees with demographics, level, department, promotion history, and attrition data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 5000
gender = np.random.choice(['Male', 'Female', 'Non-Binary'], n, p=[0.52, 0.44, 0.04])
ethnicity = np.random.choice(['White', 'Asian', 'Hispanic', 'Black', 'Other'], n, p=[0.45, 0.22, 0.16, 0.12, 0.05])
age = np.clip(np.random.normal(36, 9, n).astype(int), 22, 62)
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'Finance', 'Operations', 'HR', 'Product'],
                                 n, p=[0.22, 0.18, 0.12, 0.12, 0.14, 0.10, 0.12])

# Level distribution with representation gaps at senior levels
level_probs = {
    'Male':      [0.20, 0.28, 0.26, 0.16, 0.10],
    'Female':    [0.28, 0.30, 0.24, 0.12, 0.06],
    'Non-Binary':[0.30, 0.32, 0.22, 0.10, 0.06],
}
levels_list = ['Junior', 'Mid', 'Senior', 'Lead', 'Director']
levels = [np.random.choice(levels_list, p=level_probs[g]) for g in gender]

tenure = np.clip(np.random.exponential(4, n), 0.5, 20).round(1)
promoted = (np.random.random(n) < np.where(gender == 'Female', 0.12, np.where(gender == 'Non-Binary', 0.10, 0.16))).astype(int)
attrited = (np.random.random(n) < np.where(gender == 'Female', 0.14, np.where(gender == 'Non-Binary', 0.16, 0.11))).astype(int)

df = pd.DataFrame({
    'EmployeeID': [f'E{i:05d}' for i in range(n)],
    'Gender': gender, 'Ethnicity': ethnicity, 'Age': age,
    'Department': departments, 'Level': np.array(levels),
    'TenureYears': tenure, 'PromotedThisYear': promoted, 'Attrited': attrited,
})
df['AgeBand'] = pd.cut(df['Age'], bins=[20, 30, 40, 50, 65], labels=['20-30', '30-40', '40-50', '50+'])

print(f'Dataset: {df.shape}')
print(f'Gender: {df["Gender"].value_counts().to_dict()}')
print(f'Ethnicity: {df["Ethnicity"].value_counts().to_dict()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nLevel distribution:\n{df["Level"].value_counts()}')

## Representation by Level

In [ ]:
level_order = ['Junior', 'Mid', 'Senior', 'Lead', 'Director']
rep = df.groupby(['Level', 'Gender']).size().unstack(fill_value=0)
rep = rep.reindex(level_order)
rep_pct = rep.div(rep.sum(axis=1), axis=0).round(3)

print('Gender Representation by Level (%):')
print((rep_pct * 100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rep_pct.plot.bar(ax=axes[0], stacked=True, edgecolor='black')
axes[0].set_title('Gender Representation by Level')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Gender')

eth_rep = df.groupby(['Level', 'Ethnicity']).size().unstack(fill_value=0).reindex(level_order)
eth_pct = eth_rep.div(eth_rep.sum(axis=1), axis=0)
eth_pct.plot.bar(ax=axes[1], stacked=True, edgecolor='black')
axes[1].set_title('Ethnicity Representation by Level')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Ethnicity', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'representation_level.png'), dpi=100, bbox_inches='tight')
plt.show()

## Pipeline Leakage Analysis

In [ ]:
pipeline = df.groupby('Gender')['Level'].value_counts().unstack(fill_value=0).T.reindex(level_order)
pipeline_pct = pipeline.div(pipeline.iloc[0]) * 100  # Index to Junior = 100%
print('Pipeline Index (Junior = 100%):')
print(pipeline_pct.round(1))

fig, ax = plt.subplots(figsize=(10, 6))
for col in pipeline_pct.columns:
    ax.plot(pipeline_pct.index, pipeline_pct[col], marker='o', label=col, linewidth=2)
ax.set_title('Pipeline Leakage: Retention Index by Level (Junior = 100%)')
ax.set_ylabel('Index (Junior = 100)')
ax.legend(title='Gender')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pipeline_leakage.png'), dpi=100, bbox_inches='tight')
plt.show()

## Promotion Equity

In [ ]:
promo_gender = df.groupby('Gender').agg(
    employees=('EmployeeID', 'count'),
    promoted=('PromotedThisYear', 'sum'),
    promo_rate=('PromotedThisYear', 'mean'),
).round(3)
print('Promotion Rate by Gender:')
print(promo_gender)

promo_eth = df.groupby('Ethnicity').agg(
    employees=('EmployeeID', 'count'),
    promo_rate=('PromotedThisYear', 'mean'),
).round(3)
print('\nPromotion Rate by Ethnicity:')
print(promo_eth.sort_values('promo_rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
promo_gender['promo_rate'].plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Promotion Rate by Gender')
axes[0].tick_params(axis='x', rotation=0)
promo_eth.sort_values('promo_rate')['promo_rate'].plot.barh(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Promotion Rate by Ethnicity')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'promotion_equity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Attrition Equity

In [ ]:
att_gender = df.groupby('Gender').agg(
    employees=('EmployeeID', 'count'),
    attrition_rate=('Attrited', 'mean'),
).round(3)
print('Attrition by Gender:')
print(att_gender)

att_eth = df.groupby('Ethnicity').agg(
    employees=('EmployeeID', 'count'),
    attrition_rate=('Attrited', 'mean'),
).round(3)
print('\nAttrition by Ethnicity:')
print(att_eth.sort_values('attrition_rate', ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
att_combo = df.groupby(['Gender', 'Ethnicity'])['Attrited'].mean().unstack()
sns.heatmap(att_combo.round(2), annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Attrition Rate: Gender × Ethnicity')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'attrition_equity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department Diversity

In [ ]:
dept_gender = df.groupby(['Department', 'Gender']).size().unstack(fill_value=0)
dept_pct = dept_gender.div(dept_gender.sum(axis=1), axis=0).round(3)
print('Gender % by Department:')
print((dept_pct * 100).round(1))

fig, ax = plt.subplots(figsize=(10, 6))
dept_pct.plot.barh(ax=ax, stacked=True, edgecolor='black')
ax.set_title('Gender Composition by Department')
ax.legend(title='Gender')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dept_diversity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Representation drops sharply at senior levels** for women and non-binary employees — the "broken rung" and "leaky pipeline" are visible
- **Promotion rates** are lower for women and non-binary employees, partially explaining the senior-level gap
- **Attrition is higher** for underrepresented groups — retention efforts should be targeted
- **Engineering** tends to be more male-dominated; **HR and Marketing** show better gender balance
- **Intersectional analysis** (gender × ethnicity) reveals compounding disadvantages for certain groups
- The pipeline leakage chart is the most actionable view — it shows exactly where representation is lost

## Limitations
- Synthetic data with programmed disparities
- Binary/simplified demographic categories
- No intersectionality beyond gender × ethnicity
- No disability, LGBTQ+, or veteran status data
- No qualitative data (inclusion surveys, belonging scores)

## How to Improve This Project
- Add inclusion survey data (belonging, psychological safety)
- Include pay equity cross-referenced with demographics
- Add hiring funnel diversity analysis
- Track D&I metrics over multiple years for trend analysis
- Perform statistical significance testing on rate differences

## Production Considerations
- Quarterly D&I dashboards for leadership and board reporting
- Automated alerts when representation falls below thresholds
- Goal-setting tied to specific pipeline stage improvements
- Integration with recruitment and promotion processes

## Common Mistakes
- Reporting only overall representation without level breakdowns
- Not analyzing both promotion and attrition (both drive representation)
- Ignoring intersectionality — aggregated groups can mask compounding disparities
- Setting diversity targets without addressing root causes (pipeline, culture, bias)

## Mini Challenge / Exercises
1. Calculate the expected number of Female Directors if promotion rates were equal across genders.
2. Which department has the largest gender representation gap at the Senior+ level?
3. Calculate the "inclusion gap": difference in attrition rate between the most and least represented groups.
4. If attrition rates equalized across genders, how many additional women would be retained annually?

## Final Summary / Key Takeaways
- D&I analytics requires examining representation, advancement, and retention together
- Pipeline leakage analysis reveals where representation is lost — not just the end state
- Promotion and attrition equity are the two key levers for improving representation
- Intersectional analysis reveals hidden disparities that aggregate numbers miss
- Regular, transparent D&I reporting builds accountability and drives progress